In this tutorial/guide, we will be exploring the MCP (Model Context Protocol) technique with Claude (this content is derived from Anthropic skilljar site)

MCP client serves as the communication bridge between your server and MCP servers. It is like an access point to all the tools that an MCP server provides. When you need to use external tools or services, the client handles all the message passing and protocol details for you.

One of MCP's key strengths is being transport agnostic which is a fancy way of saying the client and server can talk to each other using different communication methods.

The most common setup runs both the MCP client and server on the same machine where they communicate through standard input/output.

MCP clients and server can also connect over:
- HTTP
- WebSockers
- Various other network protocols

Once connected, the client and server exchange specific message types defined in the MCP specification. The main message types we will work with are:

- ListToolsRequest/ListToolsResult: client asks the server "what tools do you provide?" and gets back a list of available tools.

- CallToolRequest/CallToolResult: client asks the server to run a specific tool with certain arguments, then receives the results

Here's an example of how it will work in a real scenario:

1. A user asks "What repositories do I have?" and query is submitted to the server

2. Server realizes it needs to provide Claude with a list of available tools before making the request

3. Server asks the MCP client for tools which sends a `ListToolsRequest` to the MCP server and receives a `ListToolsResult` back

4. Now your server has everything needed to make the initial request to Claude - both the user's question and the available tools

5. Claude examines the tools and decides it needs to call one to answer the question. It responds with a tool use request.

6. Your server asks the MCP client to execute the tool Claude requested. The MCP client sends a `CallToolRequest` to the MCP server, which makes the actual request to GitHub

7. GitHub returns the repo data which flows back through the MCP server as a CallToolResult then to the MCP client, and finally to your server.

8. Your server sends the tool results back to Claude in a follow-up message. Claude now has all the info it needs to formulate a complete response

9. Finally Claude responds with the formatted answer, which your server passes back to the user.


Overall, this flow contains many steps but each component has a clear responsibility. The MCP client abstracts away the complexity of server communication, allowing us to focus on application logic.

Building an MCP server becomes much simpler when we use the official Python SDK. Instead of manually writing complex JSON schemas for tools, the SDK handles all the complexity with decorators and type hints.
In this example, we are creating an MCP server that manages documents stored in memory. The server will provide two essential tools:

1. Read document contents
2. Update them through find-and-replace operations



The following is to set up the MCP server

In [ ]:
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("DocumentMCP", log_level="ERROR")

For this implementation, documents are stored in a simple Python dictionary where keys are document IDs and values contain the document content:

```
# This is formatted as code
```



In [ ]:
docs = {
    "deposition.md": "This deposition covers the testimony of Angela Smith, P.E.",
    "report.pdf": "The report details the state of a 20m condenser tower.",
    "financials.docx": "These financials outline the project's budget and expenditure",
    "outlook.pdf": "This document presents the projected future performance of the",
    "plan.md": "The plan outlines the steps for the project's implementation.",
    "spec.txt": "These specifications define the technical requirements for the equipment"
}

The SDK transforms tool creaton from a verbose process into something clean and readable using Python decorators and type hints.

The first tool allows Claude to read any document by its ID. Here's the implementation:

In [ ]:
@mcp.tool(
    name="read_doc_contents",
    description="Read the contents of a document and return it as a string."
)
def read_document(
    doc_id: str = Field(description="Id of the document to read")
):
    if doc_id not in docs:
        raise ValueError(f"Doc with id {doc_id} not found")

    return docs[doc_id]

The `@mcp.tool` decorator automatically generates the JSON schema that Claude needs. The `Field` class from Pydantic provides parameter descriptions that help Claude understand what each argument expects

The second tool performs a simple find-and-replace operations on documents. This takes three parameters: the document ID, the text to find, and the replacement text. The implementation uses Python's built-in string `replace()` method for simplicity

In [ ]:
@mcp.tool(
    name="edit_document",
    description="Edit a document by replacing a string in the documents content with a new string."
)
def edit_document(
    doc_id: str = Field(description="Id of the document that will be edited"),
    old_str: str = Field(description="The text to replace. Must match exactly, including whitespace."),
    new_str: str = Field(description="The new text to insert in place of the old text.")
):
    if doc_id not in docs:
        raise ValueError(f"Doc with id {doc_id} not found")

    docs[doc_id] = docs[doc_id].replace(old_str, new_str)

Both tools include basic error handling to manage cases where Claude requests a document that doesn't exist. This is what the `ValueError` is for which will return a descriptive message that Claude can understand and do something.

When building MCP servers, we need a way to test the functionality without connecting to a full application. The Python MCP SDK includes a built-in browser-based inspector (similar to FASTAPI) that lets you debug and test your server in real-time.

To start the inspector, we just run the following:

In [ ]:
mcp dev mcp_server.py


This starts a development server on port 6277 and gives you a local URL to open in your browser. You will see the MCP Inspector dashboard.

Click the "Connect" button (should be on the left side) to start the MCP server. Once connected it will present a navigation bar with sections for Resources, Prompts, Tools, and other features.

To test the tools:
1. Navigate to the Tools section
2. Click "List Tools" to see all available tools
3. Select a tool to open its testing interface
4. Fill in the required parameters
5. Click "Run Tool" to execute and see results

With the interactive inspector, our development loop will be:

- make changes to the MCP server code
- test individual tools through inspector
- verify results without needing a full app setup
- debug issues in isolation

Next, let's look at how to build an MCP client.

An MCP client consists of 2 main components:

1. MCP Client - A custom class we create t make using the session easier

2. Client Session - The actual connection to the server (which is part of the MCP Python SDK)


MCP Client <--> Client Session <--> MCP Server

We need to implement 2 key methods in our client.

1. `list_tools()`
This method gets all available tools from the server


2. `call_tool()`
This method executes a specific tool on the server. We pass the tool name and input parameters (provided by Claude) to the server and return the result

The following are the code implementation for these methods



In [ ]:
async def list_tools(self) -> list[types.Tool]:
    result = await self.session().list_tools()
    return result.tools

async def call_tool(
    self, tool_name: str, tool_input: dict
) -> types.CallToolResult | None:
    return await self.session().call_tool(tool_name, tool_input)

To test the implementation, we can run the client directly. The file includes a testing harness that connects to the MCP server and calls the methods

In [ ]:
async with MCPClient(
    command="uv", args=["run", "mcp_server.py"]
) as client:
    result = await client.list_tools()
    print(result)

When we run this test, we should see our tool definitions printed out

So, now our client can list tools and call them. We can finally test the complete flow. It basically acts as a bridge between our app logic and the MCP server, making it easy to access server functionality without worrying about the nitty gritty of the underlying connection. When the main application is run and user ask Claude about a document:

1. Our code uses the client to get available tools
2. These tools are sent to Claude along with the user's question
3. Claude decides to use specific tools
4. Our code uses the client to execute that tool that Claude needs to use
5. The result is sent back to Claude who then responds to the user

Resources in MCP servers allow you to expose data to clients, similar to GET request handlers in a typical HTTP server.

Let's say you want to build a document mention feature where users can type `@document_name` to reference files. This requires 2 operations:

1. Getting a list of all available documents (for autocomplete)

2. Fetching the contents of a specific document (when mentioned)

When a user types `@`, you need to show available document. When they submit a message with a mention, you automatically inject that document's content into the prompt sent to Claude.

Resources follow a request-response pattern. You client sends a `ReadResourceRequest` with a URI, and the MCP server responds with the data. There are 2 types of resourcces:

1. Direct Resources: Static URIs that don't change, like `docs://documents`

2. Templated Resoures: URIs with parameters, like `docs://documents//{doc_id}` (Python SDK can parse parameters automatically from the URI and pass them as keyword arguments to your function)

Resources are defined using `@mcp.resource()` decorator



In [ ]:
# Direct resource

@mcp.resource(
    "docs://documents",
    mime_type="application/json"
)
def list_docs() -> list[str]:
    return list(docs.keys())


# Templated Resource
@mcp.resource(
    "docs://documents/{doc_id}",
    mime_type="text/plain"
    # mime_type gives clients a hint about what kind of data you're returning.
    # 1. application/json - Structured JSON data
    # 2. text/plain - plain text content
    # 3. any valid MIME type for diff data formats

)
def fetch_doc(doc_id: str) -> str:
    if doc_id not in docs:
        raise ValueError(f"Doc with id {doc_id} not found")
    return docs[doc_id]

# you can test the resources using the MCP inspector

After defining resources on the MCP server, your client needs a way to request and use them.

When a user wants to reference a document, e.g. typing "@report.pdf", your application uses the MCP client to fetch that resource from the server and include its contents directly in the prompt sent to Claude.



The core functionality requires a `read_resource` function in your MCP client. This function takes a URI parameter identifying which resource to fetch.

The response from the MCP server contains a contents list. You usually only need the first element which contains the actual resource data along with metadata like the MIME type.

In [ ]:
import json
from pydantic import AnyUrl
# we need the above imports in the MCP client

async def read_resource(self, uri: str) -> Any:
    result = await self.session().read_resource(AnyUrl(uri))
    resource = result.contents[0]


    # the following handle the different types of content data
    if isinstance(resource, types.TextResourceContents):
        if resource.mimeType == "application/json":
            return json.loads(resource.text)

        return resource.text

Next, we can also use Prompts in MCP servers to define pre-built, high-quality instructions that MCP clients can use instead of writing their own prompts from scratch

Prompts define a set of user and assistant messages that clients can use directly. When a client requests a prompt, your server returns a list of messages that can be sent straight to Claude. The structure looks like this:

- Define prompts using the `@mcp.prompt() decorator`
- Add a name and description for each prompt
- Return a list of messages that form the complete prompt
- These prompts should be high quality, well tested an relevant to the MCP server's purpose.

Here's how:

In [ ]:
from mcp.server.fastmcp import base


@mcp.prompt(
    name="format",
    description="Rewrites the contents of the document in Markdown format."
)
def format_document(
    doc_id: str = Field(description="Id of the document to format")
) -> list[base.Message]:
    prompt = f"""
Your goal is to reformat a document to be written with markdown syntax.

The id of the document you need to reformat is:

{doc_id}


Add in headers, bullet points, tables, etc as necessary. Feel free to add in extra formatting.
Use the 'edit_document' tool to edit the document. After the document has been reformatted...
"""

    return [
        base.UserMessage(prompt)
    ]
